# How to work with COSMO data

---

COSMO data are available between 2017 and 2023.

The forecasts are computed once every 3 hours and have a lead time of 34 hours. 

In order to get hourly data, we always take COSMO forecast for lead times 0,1 and 2. 

This gives us the most precise hourly forecast data we can get from the model.

Each file contains forecast for 1 day in 1-hourly resolution.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

### Prepare utility functions to load pre-processed COSMO data

COSMO has been a single model until 17.9.2020, and an ensemble of 11 forecasts since 18.9.2020.

Therefore, the <u>files until 17.9.2020</u> have the following shape:

(224, 320, 24, 8) -> (grid_height, grid_width, hours, num_variables)

and <u>files from 18.9.2020</u> have the following shape:

(224, 320, 24, 8, 2) -> (grid_height, grid_width, hours, num_variables, ensemble_mean_and_stddev), where ensemble_mean is at location [0] and emsemble_stddev at dimension [1]

In [2]:
import h5py

# The dictionary of features available in each file
feature_name_to_index = {
    'tp': 0,            # Total Precipitaton        [mm]
    'ws': 1,            # Wind Speed                [m/s]
    'wdir': 2,          # Wind Direction            [°]
    '2t': 3,            # 2m Temperature            [°C]
    '2d': 4,            # 2m Dewpoint Temperature   [°C]
    'nswrs': 5,         # Net Shorware Radiation    [W/m^2]
    'nlwrs': 6,         # Net Longwave Radiation    [W/m^2]
    'vis': 7            # Visibility                (not sure what exactly this is for, I am not using it)
}

# Function to load the forecasts array for 1 day (24 hours) from file
# Already takes care of handling whether the COSMO forecast was a single prediction or an ensemble
# In case of an ensemble, only mean is returned
# The array is additionally reshaped so that hours are at dimension 0
# This means that this function always returns an array of shape (24, 224, 320, 8)
def load_cosmo_grid(data_path : str, year : int, month : int, day : int) -> np.array:
    # Load data from file
    hf = h5py.File(data_path + '%04i/%04i%02i%02i.h5' % (year, year, month, day), 'r')
    hf.keys()
    cosmo_grid = np.array(hf.get('cosmo_grid'))

    # Take only mean prediction
    if len(cosmo_grid.shape) == 5:
        cosmo_grid = cosmo_grid[..., 0].transpose(2, 0, 1, 3)
    else:
        cosmo_grid = cosmo_grid.transpose(2, 0, 1, 3)
    cosmo_grid[:, :, :, [3, 4]] -= 273.15  # Convert temperatures from Kelvins into Degrees celsius
    
    return cosmo_grid 

### Load data examples and print their shapes

Note that independently of whether COSMO or COSMO-ENS is used, the function load_cosmo_grid returns the same arrays (data internally pre-processed).

In [3]:
data_path = '/Users/lukovic/data/COSMO_FromCirrus/'

In [4]:
day_grid = load_cosmo_grid(data_path, 2020, 9, 17)
print(day_grid.shape)                                   # Before ensemble COSMO, dimensions reshuffled to have hours first
day_grid = load_cosmo_grid(data_path, 2020, 9, 18)  
print(day_grid.shape)                                   # After ensemble COSMO, dimensions reshuffled to have hours first, only mean extracted

(24, 224, 320, 8)
(24, 224, 320, 8)


### COSMO model corrensponding DHM (Digital Height Model)

We extract COSMO forecasts on the grid which corresponds to a DHM which we have available.

This DHM has resolution of 224 (height) x 320 (width) pixels.

The DHM is provided as a separate file with shape (height, width, 3), where:
height_map[..., 0] - latitudes
height_map[..., 1] - longitudes
height_map[..., 2] - elevations

In [5]:
import pickle

with open(data_path + 'height_map.pkl', 'rb') as f:
    height_map = pickle.load(f)

In [6]:
print(height_map.shape)

print(height_map[..., 0])
print(height_map[..., 1])
print(height_map[..., 2])

(224, 320, 3)
[[45.646352 45.646868 45.647384 ... 45.719004 45.718944 45.71888 ]
 [45.656336 45.656856 45.657372 ... 45.729004 45.728944 45.72888 ]
 [45.666324 45.66684  45.667356 ... 45.739004 45.738944 45.73888 ]
 ...
 [47.853256 47.853796 47.854332 ... 47.928964 47.9289   47.928832]
 [47.86324  47.86378  47.864316 ... 47.938964 47.9389   47.938832]
 [47.873228 47.873768 47.874304 ... 47.948964 47.9489   47.948832]]
[[ 5.93682   5.951101  5.965383 ... 10.472544 10.486863 10.501182]
 [ 5.936078  5.950362  5.964647 ... 10.47263  10.486952 10.501274]
 [ 5.935336  5.949623  5.96391  ... 10.472717 10.487041 10.501365]
 ...
 [ 5.76615   5.781029  5.795908 ... 10.492438 10.50736  10.522282]
 [ 5.765345  5.780226  5.795108 ... 10.492532 10.507457 10.522381]
 [ 5.76454   5.779424  5.794309 ... 10.492626 10.507553 10.522481]]
[[ 382.64057541  775.01557541 1211.64057541 ...  689.14057541
   796.76557541  934.89057541]
 [ 356.64057541  695.64057541 1127.64057541 ...  560.89057541
   736.76557541

### Plot an example of the COSMO data

In [7]:
import plotly.express as px

# Pick 0-th hour of the day (midnight)
# Flip height dimension (otherwise Swiss Grid would be plotted upside-down)
# Pick 3-rd variable in the array (index 3 is 2m temperature as per dictionary in one of the cells above)
px.imshow(day_grid[0, ::-1, :, 3])

In [8]:
# Plot the corresponding DHM elevation image
# Don't forget to flip height dimension (otherwise Swiss Grid would be plotted upside-down)
px.imshow(height_map[::-1, :, 2])

### Plot the DHM

Here we use Scattergeo function which allows us to plot data for particular GPS locations, and not necessarily as an image. 

This way we can demonstrate that the height_map array really stores GPS coordinates with corresponding terrain elevations.

In [9]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scattergeo(
        lon = height_map[..., 1].flatten(),
        lat = height_map[..., 0].flatten(),
        mode = 'markers',
        marker_color = height_map[..., 2].flatten(),
    ),
)
fig.update_layout(
        title = 'Digital height Model',
        geo_scope='europe',
        geo_showland=False,
        geo_projection_scale=20,
        geo_center=dict(
            lon=height_map[..., 1].flatten().mean(),
            lat=height_map[..., 0].flatten().mean()
        )
    )

fig.show()

### Locating closest point on the grid w.r.t. given GPS location

First we compute the distance map...

In [10]:
location = np.array([47.362950, 8.454470]) # Coordinates from metadata of a tree in Birmensdorf

height, width, chans = height_map.shape
hmap = height_map.reshape(-1, chans)[:, :2]
all_dists = np.sqrt(((hmap[None, :, :] - location[None, None, :])**2).sum(axis=2)) # (1, num_pts)
        
print(all_dists)

[[3.04717414 3.0350937  3.02303172 ... 2.12072919 2.13506141 2.14940159]]


Plot the distance map to visualize distance of each point in the Digital Height Model w.r.t. our target point (Birmensdorf)

In [11]:
px.imshow(all_dists.reshape(height, width).astype(np.float32)[::-1, :])

Get location of the target point (smallest distance) in the height_map array and use it to extract corresponding COSMO forecasts.

In [12]:
# Extract the index of location with minimum distance
min_idx = all_dists.argmin()

# Get data from COSMO
cosmo_grid = load_cosmo_grid(data_path, 2020, 9, 18)  
print(cosmo_grid.shape)                                  
# Reshape COSMO forecasts to flatten across spatial dimensions
cosmo_flat = cosmo_grid.reshape(cosmo_grid.shape[0], -1, cosmo_grid.shape[3])
print(cosmo_flat.shape)

# Use the min-distance index to extract COSMO variables for the desired location
cosmo_location = cosmo_flat[:, min_idx, :]
print(cosmo_location.shape)

(24, 224, 320, 8)
(24, 71680, 8)
(24, 8)


### Plot time-series example for the 24 hours at the given location, for e.g. temperature (index 3)

In [13]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=np.arange(cosmo_location.shape[0]),
        y=cosmo_location[:, 3]
    )
)
fig.update_layout(
    xaxis=dict(
        title='Time [hour]'
    ),
    yaxis=dict(
        title='2m Temperature [°C]'
    )
)
fig.show()